# NLP Exercise 1

Unigram and bigram language models trained on Wikitext-2.

This starter cell loads the dependencies and data per the PDF instructions.

In [1]:
import spacy
from datasets import load_dataset

nlp = spacy.load("en_core_web_sm")
text = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
doc = nlp(text[0]["text"])
doc

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

# unigram

In [ ]:
from collections import Counter
import math

# Reload with parser/ner disabled for speed
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

lines = [row["text"] for row in text if row["text"].strip()]

counts = Counter()
total = 0

for doc in nlp.pipe(lines, batch_size=50):
    for tok in doc:
        if tok.is_alpha:
            lemma = tok.lemma_.lower()
            counts[lemma] += 1
            total += 1

likelihood = {w: c/total for w, c in counts.items()}

## bigram 


In [4]:
from collections import Counter, defaultdict

## bigram 
counts = Counter()
after = defaultdict(Counter)            

for doc in nlp.pipe(lines, batch_size=50):
    lemmas = [tok.lemma_.lower() for tok in doc if tok.is_alpha]
    tokens = ["START"] + lemmas
    for i in range(len(tokens) - 1):
        prev = tokens[i]
        curr = tokens[i + 1]
        after[prev][curr] += 1         
        counts[prev] += 1   

bigram_likelihood = {}
for prev, nexts in after.items():
    total = counts[prev]
    bigram_likelihood[prev] = {curr: n/total for curr, n in nexts.items()}



In [5]:
import json

with open("bigram_model.json", "w") as f:
    json.dump(bigram_likelihood, f)
    

In [ ]:
import json

with open("bigram_model.json") as f:
    bigram_likelihood = json.load(f)